# Assignment 2: Gradio dashboard

You're using a notebook for this stage. If you'd rather use a script,
write `app.py` instead and delete this file. `main.py` refuses to run if
it finds both `app.py` and `app.ipynb`, so exactly one of them must exist.

Replace this cell and everything below it with your own code. When run
via `uv run python main.py app`, this notebook must build and launch a Gradio app
(a `demo` that calls `.launch()`) using only what your training stage already
produced. It must never train anything itself. Load the committed weights in
`weights/`, never whatever the training run left in `runs/`.

At minimum, the dashboard must let you:

- Upload an image and see the model's predictions at the current threshold.
- Move the two edges of your escalation band and watch the confusion matrix,
  the expected cost per 1,000 units, and the share of cases escalated to a
  human all change together.
- See the cost-against-threshold curve with your chosen operating point marked
  on it.
- See per-class performance, not just the overall number.

How you structure the code beyond that is your call to make and be able to
defend: file layout, function boundaries, naming.

In [ ]:

import json
import os
from pathlib import Path

import gradio as gr
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw
from ultralytics import YOLO

ROOT = Path.cwd().resolve()
ARTIFACTS = ROOT / "artifacts"

METRICS_FILE = ARTIFACTS / "metrics.json"
WEIGHT_FILE = ROOT / "weights" / "best.pt"

if (
    not METRICS_FILE.is_file()
    or not WEIGHT_FILE.is_file()
):
    raise FileNotFoundError(
        "Missing metrics or trained weights. "
        "Run main.py train first, then commit outputs."
    )

M = json.loads(
    METRICS_FILE.read_text(encoding="utf-8")
)

VAL = pd.read_csv(
    ARTIFACTS / "val_cases.csv"
)

TEST = pd.read_csv(
    ARTIFACTS / "test_cases.csv"
)

HEALTH = pd.read_csv(
    ARTIFACTS / "dataset_health.csv"
)

MODEL = YOLO(str(WEIGHT_FILE))

COST = M["cost_euros"]
HEAD = M["class_ids"]["head"]
HELMET = M["class_ids"]["helmet"]
SINGLE = M["single_threshold"]
NAMES = M["class_names"]


In [ ]:

def case_actions(
    data,
    lo,
    hi,
    triage=True
):
    scores = data[
        "unsafe_score"
    ].to_numpy(dtype=float)

    if not triage:
        return np.where(
            scores >= lo, 2, 0
        )

    actions = np.where(
        scores >= hi,
        2,
        np.where(
            scores < lo, 0, 1
        )
    )

    return np.where(
        data["no_evidence"].to_numpy(
            dtype=bool
        ),
        1,
        actions
    )


def measure(data, action, prior):
    y = data[
        "unsafe_gt"
    ].to_numpy(dtype=int)

    cm = np.array([
        [
            np.sum(
                (y == i) & (action == j)
            )
            for j in range(3)
        ]
        for i in (0, 1)
    ], dtype=int)

    rates = cm / cm.sum(
        axis=1,
        keepdims=True
    )

    cost_safe = np.dot(
        rates[0],
        [
            COST["tn"],
            COST["review"],
            COST["fp"]
        ]
    )

    cost_unsafe = np.dot(
        rates[1],
        [
            COST["fn"],
            COST["review"],
            COST["tp"]
        ]
    )

    expected = 1000 * (
        (1 - prior) * cost_safe +
        prior * cost_unsafe
    )

    review = (
        (1 - prior) * rates[0, 1] +
        prior * rates[1, 1]
    )

    return (
        cm,
        float(expected),
        float(review)
    )


def object_results_table():
    rows = []

    for label, benchmark in (
        M["benchmarks"].items()
    ):
        for values in benchmark[
            "per_class"
        ]:
            rows.append({
                "Model": label,
                "Class": values["class"],

                "Precision @ .25":
                    values["precision_at_0.25"],

                "Recall @ .25":
                    values["recall_at_0.25"],

                "AP50": values["AP50"],

                "mAP50–95":
                    values["mAP50_95"],

                "Test GT objects":
                    values["gt_instances"]
            })

    return pd.DataFrame(rows)


In [ ]:

def figure_confusion(cm):
    corrected = np.array([
        [
            cm[0, 0] + cm[0, 1],
            cm[0, 2]
        ],
        [
            cm[1, 0],
            cm[1, 1] + cm[1, 2]
        ]
    ])

    fig, axes = plt.subplots(
        1, 2,
        figsize=(12, 4.1),
        constrained_layout=True
    )

    chart_specs = [
        (
            axes[0],
            cm,
            ["Pass", "Review", "Flag"],
            "Before human review (test images)"
        ),
        (
            axes[1],
            corrected,
            ["Cleared", "Flagged"],
            "After review (perfect-review assumption)"
        )
    ]

    for ax, arr, colnames, title in chart_specs:
        ax.imshow(
            arr,
            cmap="Blues",
            vmin=0
        )

        ax.set_xticks(
            range(len(colnames)),
            colnames
        )

        ax.set_yticks(
            [0, 1],
            ["No bare head", "Bare head"]
        )

        ax.set_xlabel("System action")
        ax.set_ylabel("Annotated image")
        ax.set_title(title)

        for (r, c), val in np.ndenumerate(arr):
            ax.text(
                c, r, str(val),
                ha="center",
                va="center",
                fontsize=11
            )

    return fig


def figure_sweep(prior, lo, hi):
    x = np.linspace(
        0.01, 1.0, 100
    )

    costs = np.array([
        measure(
            VAL,
            case_actions(
                VAL, t, t, False
            ),
            prior
        )[1]
        for t in x
    ])

    fig, ax = plt.subplots(
        figsize=(10, 4),
        constrained_layout=True
    )

    ax.plot(
        x,
        costs,
        lw=2,
        label=(
            "One-threshold model alone "
            "(validation)"
        )
    )

    ax.axvspan(
        lo, hi,
        color="orange",
        alpha=0.12,
        label="Current review band"
    )

    ax.axvline(
        SINGLE,
        ls="--",
        color="black",
        label=(
            f"Chosen at assumed "
            f"{M['base_rate_assumed']:.0%} "
            f"prevalence: {SINGLE:.2f}"
        )
    )

    min_index = int(
        np.argmin(costs)
    )

    ax.scatter(
        [x[min_index]],
        [costs[min_index]],
        marker="o",
        s=55,
        label=(
            "Best single cutoff at "
            "selected prevalence: "
            f"{x[min_index]:.2f}"
        )
    )

    ax.set(
        xlabel=(
            "Exposed-head detection "
            "confidence threshold"
        ),
        ylabel=(
            "Expected euro cost "
            "per 1,000 images"
        ),
        title=(
            "Threshold selection uses "
            "validation, not test"
        )
    )

    ax.grid(alpha=0.2)
    ax.legend(
        fontsize=8,
        loc="best"
    )

    return fig


def refresh(low, high, prior_percent):
    lo, hi = sorted((
        float(low),
        float(high)
    ))

    prior = (
        float(prior_percent) / 100.0
    )

    cm, hybrid_cost, review_fraction = measure(
        TEST,
        case_actions(TEST, lo, hi),
        prior
    )

    _, model_cost, _ = measure(
        TEST,
        case_actions(
            TEST,
            SINGLE,
            SINGLE,
            False
        ),
        prior
    )

    nothing_cost = 1000 * (
        prior * COST["fn"] +
        (1 - prior) * COST["tn"]
    )

    all_human_cost = (
        1000 * COST["review"]
    )

    reviews_day = (
        review_fraction *
        M["cases_per_day"]
    )

    minutes = (
        reviews_day *
        M["review_seconds"] / 60
    )

    full_time_equiv = (
        minutes /
        M["fte_minutes_per_day"]
    )

    warning = (
        "**Threshold order corrected for "
        "this calculation** "
        "(lower must be <= upper).  "
        if low > high
        else ""
    )

    status = (
        "Within allocated reviewer time"
        if minutes <=
        M["review_minutes_available"]
        else (
            "OVER the allocated reviewer time: "
            "this policy is not operationally feasible"
        )
    )

    overview = (
        f"{warning}"
        f"**Current pass/review/flag thresholds:** "
        f"{lo:.3f} / {hi:.3f}  \n"

        f"**Expected cost:** €{hybrid_cost:,.0f} "
        f"per 1,000 images at "
        f"**{prior_percent:.0f}%** "
        f"assumed prevalence  \n"

        f"**Review share:** "
        f"{review_fraction:.1%} "
        f"(prevalence-adjusted); "

        f"**reviews/day:** {reviews_day:,.1f}; "
        f"**supervisor minutes/day:** {minutes:,.1f}; "
        f"**FTE equivalents:** {full_time_equiv:.2f}  \n"

        f"**Capacity:** {status} "
        f"(allocated: "
        f"{M['review_minutes_available']} "
        f"minutes/day).  \n"

        "**Review calculation assumes a human "
        "checks every referred image correctly.** "
        "The displayed confusion matrices use "
        "raw held-out test counts; expected costs "
        "and review shares are reweighted to "
        "the prevalence slider."
    )

    alternatives = pd.DataFrame([
        [
            "Do nothing (always pass)",
            nothing_cost,
            "0%"
        ],
        [
            "Human checks every image",
            all_human_cost,
            "100%"
        ],
        [
            f"Model alone "
            f"(validated cutoff {SINGLE:.2f})",
            model_cost,
            "0%"
        ],
        [
            "Hybrid at current thresholds",
            hybrid_cost,
            f"{review_fraction:.1%}"
        ]
    ], columns=[
        "Strategy",
        "€/1,000 images",
        "Share reviewed"
    ])

    alternatives[
        "€/1,000 images"
    ] = alternatives[
        "€/1,000 images"
    ].round(2)

    return (
        overview,
        figure_confusion(cm),
        figure_sweep(prior, lo, hi),
        alternatives
    )


In [ ]:

def inspect_image(
    image,
    lower,
    upper
):
    if image is None:
        return (
            None,
            "Upload an image to obtain "
            "predictions from the committed model."
        )

    lo, hi = sorted((
        float(lower),
        float(upper)
    ))

    rgb = image.convert("RGB")

    result = MODEL.predict(
        source=rgb,
        imgsz=M["imgsz"],
        conf=M["inference_conf"],
        iou=M["nms_iou"],
        max_det=300,
        verbose=False
    )[0]

    boxes = result.boxes

    ids = (
        boxes.cls.cpu()
        .numpy().astype(int)
    )
    scores = (
        boxes.conf.cpu().numpy()
    )
    coords = (
        boxes.xyxy.cpu().numpy()
    )

    head_score = max(
        (
            float(s)
            for cid, s in zip(ids, scores)
            if cid == HEAD
        ),
        default=0.0
    )

    helmet_score = max(
        (
            float(s)
            for cid, s in zip(ids, scores)
            if cid == HELMET
        ),
        default=0.0
    )

    no_evidence = (
        max(head_score, helmet_score)
        < M["evidence_conf"]
    )

    action = (
        "REVIEW — neither head nor helmet "
        "has reliable evidence"
        if no_evidence else

        "FLAG — potential missing hard hat"
        if head_score >= hi else

        "PASS — no sufficiently confident "
        "exposed-head signal"
        if head_score < lo else

        "REVIEW — borderline exposed-head signal"
    )

    canvas = rgb.copy()
    draw = ImageDraw.Draw(canvas)

    for cid, score, (
        x1, y1, x2, y2
    ) in zip(ids, scores, coords):

        visible = (
            score >= lo
            if cid == HEAD
            else score >= 0.25
        )

        if not visible:
            continue

        color = (
            "#DC2626"
            if cid == HEAD and score >= hi
            else "#D97706"
            if cid == HEAD
            else "#2563EB"
        )

        draw.rectangle(
            (
                float(x1),
                float(y1),
                float(x2),
                float(y2)
            ),
            outline=color,
            width=3
        )

        label = (
            f"{NAMES[int(cid)]}: {score:.2f}"
        )

        draw.text(
            (
                float(x1) + 3,
                max(0, float(y1) - 15)
            ),
            label,
            fill=color,
            stroke_width=1,
            stroke_fill="white"
        )

    explanation = (
        f"### {action}\n"

        f"**Maximum exposed-head confidence:** "
        f"{head_score:.3f}. "

        f"**Maximum helmet confidence:** "
        f"{helmet_score:.3f}.  \n"

        f"**Current review interval:** "
        f"{lo:.3f}–{hi:.3f}. "

        f"**No-evidence review override:** "
        f"{'ON' if no_evidence else 'off'}.  \n"

        "*This is a screening signal, not a "
        "determination that a worker is safe "
        "or compliant. Detector confidence is "
        "not a calibrated probability.*"
    )

    return canvas, explanation


In [ ]:

with gr.Blocks(
    title="Hard Hat Safety — Cost-Aware Screening"
) as demo:

    gr.Markdown("""
# Hard Hat Safety — Cost-Aware Screening

**YOLO11s · construction-worker images · pass / human review / flag**

Thresholds were selected on validation. The interactive
confusion matrix uses held-out test images.

No training and no Roboflow download happen at app startup.
""")

    with gr.Row():
        lower = gr.Slider(
            0, 1,
            value=M["low_threshold"],
            step=0.005,
            label=(
                "Lower threshold — below this: "
                "PASS (unless no evidence)"
            )
        )

        upper = gr.Slider(
            0, 1,
            value=M["high_threshold"],
            step=0.005,
            label=(
                "Upper threshold — at/above "
                "this: FLAG"
            )
        )

        prevalence = gr.Slider(
            1, 60,
            value=M["base_rate_assumed"] * 100,
            step=1,
            label=(
                "Assumed deployment prevalence "
                "of an exposed head (%)"
            )
        )

    overview = gr.Markdown()

    with gr.Row():
        matrix_plot = gr.Plot(
            label="Live decision confusion matrices"
        )

        threshold_plot = gr.Plot(
            label="Cost vs confidence threshold"
        )

    comparisons = gr.Dataframe(
        label=(
            "Business strategy comparison "
            "(test-based, prevalence-adjusted)"
        ),
        interactive=False
    )

    gr.Markdown(
        "### Inspect a new image"
    )

    with gr.Row():
        uploaded = gr.Image(
            type="pil",
            label="Upload a construction-site image"
        )

        predicted = gr.Image(
            type="pil",
            label=(
                "Model detections at "
                "current thresholds"
            ),
            interactive=False
        )

    message = gr.Markdown(
        "Upload an image to begin."
    )

    gr.Markdown(
        "### Detection results on the "
        "held-out test split"
    )

    benchmark_table = (
        object_results_table()
    )

    gr.Dataframe(
        value=benchmark_table,
        label=(
            "Per-class results, identical "
            "test images and matching protocol"
        ),
        interactive=False
    )

    aggregate = pd.DataFrame([
        {
            "Model": name,
            "mAP50": res["AP50"],
            "mAP50–95": res["mAP50_95"]
        }
        for name, res in M[
            "benchmarks"
        ].items()
    ])

    gr.Dataframe(
        value=aggregate,
        label="Overall detection AP comparisons",
        interactive=False
    )

    gr.Markdown(f"""
**Class of interest:** `{NAMES[HEAD]}`
(possible unprotected head).

**Test images:** {len(TEST):,}.
**Validation images:** {len(VAL):,}.

**Observed test exposed-head image prevalence:**
{M['dataset_test_base_rate']:.1%}.

**COCO zero-shot baseline:** Only the shared
`person` category is mapped. COCO has no directly
equivalent `head` or `helmet` classes.

**Conservative override:** If the highest detected
head and helmet confidences are both below
{M['evidence_conf']:.2f}, route the case to review.

A missing detection is not proof of safety.
Human-review cost estimates assume an error-free
reviewer. The assigned euro amounts are scenarios,
not measured injury costs.
""")

    dependencies = [
        lower,
        upper,
        prevalence
    ]

    destinations = [
        overview,
        matrix_plot,
        threshold_plot,
        comparisons
    ]

    demo.load(
        refresh,
        inputs=dependencies,
        outputs=destinations
    )

    for ctrl in dependencies:
        ctrl.change(
            refresh,
            inputs=dependencies,
            outputs=destinations
        )

    for ctrl in [
        uploaded,
        lower,
        upper
    ]:
        ctrl.change(
            inspect_image,
            inputs=[
                uploaded,
                lower,
                upper
            ],
            outputs=[
                predicted,
                message
            ]
        )


if __name__ == "__main__":
    demo.launch(
        server_name=os.getenv(
            "APP_HOST",
            "127.0.0.1"
        ),
        server_port=int(
            os.getenv("APP_PORT", "7860")
        ),
        share=False,
        show_error=True
    )
